# 01 — Red vial y servidor OSRM

**CC2017 Modelación y Simulación — Ciclo 2, 2026**
Proyecto 1: impacto del cierre de una vía sobre el tráfico de las zonas 10, 15 y 16
de la Ciudad de Guatemala.

Este notebook deja lista la **infraestructura** sobre la que corre todo lo demás:

1. Descarga el extracto de OpenStreetMap de Guatemala.
2. Compila el grafo de OSRM y levanta el servidor local.
3. Verifica que el servidor local enruta igual que el público.
4. Caracteriza la red del área y ubica las dos vías candidatas a cerrar.

No hay simulación aquí y no hay ninguna variable aleatoria. Esa parte empieza en
el notebook 03. Lo que sale de aquí es determinista a propósito: la red es el
escenario fijo sobre el que después se echa a andar el tráfico.

> **Por qué un OSRM local y no el público.** El servidor demo
> `router.project-osrm.org` rechaza el parámetro `exclude`, así que no permite
> cerrar calles. Con una instancia propia el cierre se hace **en el grafo** y
> OSRM recalcula el óptimo real sobre la red mutilada. Ver
> [`estructura-proyecto.md` §6](../estructura-proyecto.md).

## 0. Configuración

In [1]:
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import osrm
import red

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print(f"raíz del proyecto : {config.RAIZ}")
print(f"caja de estudio   : {config.BBOX}   (sur, oeste, norte, este)")
print(f"semilla global    : {config.SEMILLA:,}")

raíz del proyecto : /Users/fabianprado/Documents/projects/2026 - 2 /modelacion/proyecto
caja de estudio   : (14.575, -90.53, 14.63, -90.455)   (sur, oeste, norte, este)
semilla global    : 20,172,026


## 1. Área de estudio

Las tres zonas y los puntos ancla que se usan a lo largo del proyecto. Todo esto
vive en `src/config.py`: si una coordenada hay que cambiarla, se cambia ahí y no
en el notebook.

In [2]:
zonas = pd.DataFrame([
    {"clave": k, "nombre": v["nombre"], "lat": v["centro"][0],
     "lon": v["centro"][1], "peso destino": v["peso"]}
    for k, v in config.ZONAS.items()
])
display(zonas)

assert abs(zonas["peso destino"].sum() - 1.0) < 1e-9, "los pesos de destino deben sumar 1"

print("\nPuntos ancla (geocodificados con Nominatim):")
for k, (la, lo) in config.PUNTOS.items():
    print(f"  {k:15s} {la:9.4f}, {lo:10.4f}")

,clave,nombre,lat,lon,peso destino
0,z10,Zona 10 (Zona Viva / Oakland),14.5990,-90.5120,0.40
1,z15,Zona 15 (Vista Hermosa),14.6047,-90.4896,0.35
2,z16,Zona 16 (Cayalá / Landívar),14.6084,-90.4865,0.25



Puntos ancla (geocodificados con Nominatim):
  uvg               14.6047,   -90.4896
  cayala            14.6084,   -90.4865
  proceres          14.5897,   -90.5051
  reforma           14.6060,   -90.5153
  vista_hermosa     14.6060,   -90.5016


## 2. Extracto de OpenStreetMap

Geofabrik publica un extracto de Guatemala de ~131 MB que se actualiza a diario.
La descarga es idempotente: si el archivo ya está, no se vuelve a bajar.

El `.pbf` **no va al repositorio** (está en `.gitignore`). Cada quien lo baja una
vez en su máquina.

In [3]:
pbf = red.descargar_pbf()
print(f"\n{pbf.name}: {pbf.stat().st_size / 1e6:.0f} MB")

ya existe: /Users/fabianprado/Documents/projects/2026 - 2 /modelacion/proyecto/data/osm/guatemala-latest.osm.pbf (131 MB)

guatemala-latest.osm.pbf: 131 MB


## 3. Grafo de OSRM

El pipeline MLD tiene tres pasos, y la gracia está en que solo el último es
barato:

| Paso | Qué hace | Costo | ¿Se repite? |
|---|---|---|---|
| `osrm-extract` | Lee el `.pbf` y aplica el perfil de auto | minutos | no |
| `osrm-partition` | Particiona el grafo en celdas multinivel | minutos | no |
| `osrm-customize` | Asigna pesos a las aristas | **segundos** | **sí, por escenario** |

Cerrar una calle es reasignar pesos, o sea que solo hay que volver a correr
`osrm-customize`. Por eso el proyecto puede permitirse decenas de escenarios en
vez de dos.

La celda siguiente **no construye nada por su cuenta**: revisa el estado y
dice qué hacer. La construcción tarda varios minutos y conviene verla correr en
una terminal, no enterrada en la salida de un notebook.

In [4]:
def docker_activo() -> bool:
    try:
        subprocess.run(["docker", "ps"], capture_output=True, timeout=10, check=True)
        return True
    except Exception:
        return False


base = config.PBF_NOMBRE.replace(".osm.pbf", "")
grafo = config.OSM / f"{base}.osrm.mldgr"   # lo produce osrm-customize

if not docker_activo():
    print("Docker no está corriendo. Abrir Docker Desktop y volver a ejecutar esta celda.")
elif grafo.exists():
    print(f"Grafo ya construido: {grafo.name} ({grafo.stat().st_size / 1e6:.0f} MB)")
else:
    print("Falta construir el grafo. En una terminal, desde la raíz del proyecto:\n")
    print("    ./osrm/construir.sh\n")
    print("La imagen trae build nativo arm64, así que en Apple Silicon corre")
    print("sin emulación. Si aun así el pull falla:\n")
    print('    PLATAFORMA="--platform linux/amd64" ./osrm/construir.sh\n')
    print("Tarda varios minutos. Es una sola vez.")

Falta construir el grafo. En una terminal, desde la raíz del proyecto:

    ./osrm/construir.sh

En Apple Silicon, si el `docker pull` falla por arquitectura:

    PLATAFORMA="--platform linux/amd64" ./osrm/construir.sh

Tarda varios minutos. Es una sola vez.


## 4. Servidor

Con el grafo listo, el servidor se levanta con Docker Compose y queda escuchando
en `localhost:5000`.

```bash
docker compose -f osrm/docker-compose.yml up -d
```

Si el local no está disponible, el notebook **sigue** contra el servidor público:
todo lo de este notebook (caracterizar la red, medir rutas base) funciona igual.
Lo único que exige el local son los cierres, y eso es el notebook 02.

In [5]:
if osrm.disponible(config.OSRM_LOCAL):
    SERVIDOR = config.OSRM_LOCAL
    print(f"OSRM local activo en {SERVIDOR}")
else:
    SERVIDOR = config.OSRM_PUBLICO
    print(f"OSRM local no responde — usando el público: {SERVIDOR}")
    print("Recordatorio: los cierres del notebook 02 SÍ requieren el servidor local.")

assert osrm.disponible(SERVIDOR), "ningún servidor OSRM responde"

OSRM local no responde — usando el público: https://router.project-osrm.org
Recordatorio: los cierres del notebook 02 SÍ requieren el servidor local.


## 5. Validación de la red

Antes de creerle a cualquier número hay que comprobar que OSRM entiende la red
de Guatemala. Dos pruebas:

1. **Las rutas son razonables.** El cociente entre la distancia por carretera y
   la distancia en línea recta debe estar entre 1 y ~2. Un cociente cercano a 1
   sería imposible en una ciudad con barrancos; uno por arriba de 3 delataría
   una red rota.
2. **El local coincide con el público.** Si el grafo local está bien construido,
   ambos servidores deben dar prácticamente lo mismo sobre la red abierta.

In [6]:
import math


def haversine_km(a, b):
    R = 6371.0088
    p1, p2 = math.radians(a[0]), math.radians(b[0])
    dp, dl = p2 - p1, math.radians(b[1] - a[1])
    h = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))


pares = [
    ("uvg", "proceres"), ("uvg", "reforma"), ("cayala", "proceres"),
    ("cayala", "reforma"), ("vista_hermosa", "cayala"), ("reforma", "uvg"),
]

filas = []
for o, d in pares:
    po, pd_ = config.PUNTOS[o], config.PUNTOS[d]
    r = osrm.ruta(po, pd_, servidor=SERVIDOR)
    recta = haversine_km(po, pd_)
    filas.append({
        "origen": o, "destino": d,
        "recta_km": recta,
        "ruta_km": r["distancia_m"] / 1000,
        "rodeo": r["distancia_m"] / 1000 / recta,
        "flujo_libre_min": r["duracion_s"] / 60,
        "nodos": len(r["nodos"]),
    })

base_rutas = pd.DataFrame(filas)
display(base_rutas.round(3))

# Cota generosa a propósito: en esta ciudad los rodeos son enormes y eso es
# real, no un síntoma. Lo que la aserción descarta es una red rota, donde una
# ruta se iría por el otro lado del país.
assert base_rutas["rodeo"].between(1.0, 6.0).all(), "hay rodeos imposibles: red sospechosa"

peor = base_rutas.loc[base_rutas["rodeo"].idxmax()]
print(f"\nRodeo mediano: {base_rutas['rodeo'].median():.2f}× la línea recta.")
print(f"Peor caso: {peor['origen']} → {peor['destino']} — "
      f"{peor['recta_km']:.2f} km en línea recta, {peor['ruta_km']:.2f} km por calle "
      f"({peor['rodeo']:.2f}×).")
print("\nEse 4× NO es un error de la red: es la geografía de la ciudad. Los")
print("barrancos separan la zona 15 de la zona 10 y obligan a rodear por unos")
print("pocos cruces. Es exactamente la razón por la que cerrar una vía aquí")
print("debería doler: la red ya está trabajando con muy poca holgura.")
print("\nLos minutos son de FLUJO LIBRE: no son tiempos reales de viaje.")
print("El proyecto solo usa cocientes entre escenarios, nunca estos valores en crudo.")

,origen,destino,recta_km,ruta_km,rodeo,flujo_libre_min,nodos
0,uvg,proceres,2.359,9.463,4.012,13.443,393
1,uvg,reforma,2.769,8.070,2.914,11.768,330
2,cayala,proceres,2.886,8.559,2.966,12.750,325
3,cayala,reforma,3.110,7.166,2.304,11.075,262
4,vista_hermosa,cayala,1.647,3.755,2.280,4.843,174
5,reforma,uvg,2.769,5.558,2.007,8.960,218



Rodeo mediano: 2.61× la línea recta.
Peor caso: uvg → proceres — 2.36 km en línea recta, 9.46 km por calle (4.01×).

Ese 4× NO es un error de la red: es la geografía de la ciudad. Los
barrancos separan la zona 15 de la zona 10 y obligan a rodear por unos
pocos cruces. Es exactamente la razón por la que cerrar una vía aquí
debería doler: la red ya está trabajando con muy poca holgura.

Los minutos son de FLUJO LIBRE: no son tiempos reales de viaje.
El proyecto solo usa cocientes entre escenarios, nunca estos valores en crudo.


In [7]:
# Comparación local vs. público, solo si ambos están arriba.
if SERVIDOR == config.OSRM_LOCAL and osrm.disponible(config.OSRM_PUBLICO):
    comp = []
    for o, d in pares:
        rl = osrm.ruta(config.PUNTOS[o], config.PUNTOS[d], servidor=config.OSRM_LOCAL)
        rp = osrm.ruta(config.PUNTOS[o], config.PUNTOS[d], servidor=config.OSRM_PUBLICO)
        comp.append({
            "par": f"{o}→{d}",
            "local_s": rl["duracion_s"], "publico_s": rp["duracion_s"],
            "dif_%": 100 * (rl["duracion_s"] - rp["duracion_s"]) / rp["duracion_s"],
        })
    comp = pd.DataFrame(comp)
    display(comp.round(2))
    peor = comp["dif_%"].abs().max()
    print(f"Discrepancia máxima: {peor:.2f}%")
    assert peor < 15, "el grafo local difiere demasiado del público — revisar la construcción"
else:
    print("Comparación omitida: hace falta que ambos servidores estén disponibles.")

Comparación omitida: hace falta que ambos servidores estén disponibles.


## 6. La red del área

Cuántos kilómetros de cada clase de vía hay dentro de la caja. Esto caracteriza
el sistema para la sección de descripción del informe y, de paso, confirma que
la caja encierra lo que se cree.

In [8]:
ways = red.red_del_area()

resumen = Counter()
kms = Counter()
for w in ways:
    resumen[w["highway"]] += 1
    kms[w["highway"]] += red.largo_km(w["geom"])

tabla_red = (pd.DataFrame([
        {"clase": h, "ways": resumen[h], "km": kms[h]}
        for h in resumen
    ])
    .sort_values("km", ascending=False)
    .reset_index(drop=True))
tabla_red["% km"] = 100 * tabla_red["km"] / tabla_red["km"].sum()
display(tabla_red.round(2))

print(f"\nTotal: {len(ways):,} ways · {sum(kms.values()):.1f} km conducibles")
print(f"Vías estructurantes (motorway/trunk/primary): "
      f"{sum(kms[h] for h in ('motorway', 'trunk', 'primary')):.1f} km")

,clase,ways,km,% km
0,residential,3475,443.68,59.56
1,tertiary,594,105.96,14.22
2,primary,458,74.97,10.06
3,secondary,491,70.91,9.52
4,trunk,99,22.26,2.99
5,primary_link,182,11.78,1.58
6,trunk_link,72,5.95,0.80
7,secondary_link,95,5.27,0.71
8,unclassified,46,4.16,0.56



Total: 5,512 ways · 744.9 km conducibles
Vías estructurantes (motorway/trunk/primary): 97.2 km


## 7. Las dos vías candidatas

Los nombres son literalmente los de la etiqueta `name` de OpenStreetMap y están
verificados contra Overpass. Una tilde de más aquí y la selección se queda vacía.

Nótese que ambas están mapeadas como **calzadas separadas por sentido**
(`oneway=yes`): cerrar "la calle" significa cerrar los ways de los dos sentidos,
y la selección por nombre ya los toma todos.

In [9]:
candidatas = {}
for clave in ("vista_hermosa", "reforma"):
    esc = config.ESCENARIOS[clave]
    ws = red.ways_con_nodos(esc["nombres_osm"])
    pares_nodo = red.pares_de_nodos(ws)
    candidatas[clave] = ws
    print(f"\n{esc['etiqueta']}")
    print(f"  nombres OSM        : {esc['nombres_osm']}")
    print(f"  ways               : {len(ws)}")
    print(f"  nodos              : {sum(len(w['nodos']) for w in ws)}")
    print(f"  segmentos dirigidos: {len(pares_nodo)}")
    print(f"  km                 : {sum(red.largo_km(w['geom']) for w in ws):.2f}")
    print(f"  clases             : {sorted(set(w['highway'] for w in ws))}")
    print(f"  hipótesis          : {esc['hipotesis']}")
    assert ws, f"la selección de {clave} salió vacía — revisar el nombre OSM"


Cierre Boulevard Vista Hermosa
  nombres OSM        : ['Boulevard Vista Hermosa']
  ways               : 23
  nodos              : 227
  segmentos dirigidos: 408
  km                 : 8.68
  clases             : ['primary']
  hipótesis          : Corredor crítico: espina primary de la zona 15 encajonada entre barrancos, con pocas alternativas paralelas. Se espera un Δ% grande y muy asimétrico según el par origen-destino.



Cierre Avenida Reforma (calzada principal)
  nombres OSM        : ['Avenida Reforma']
  ways               : 36
  nodos              : 117
  segmentos dirigidos: 162
  km                 : 4.60
  clases             : ['primary', 'tertiary']
  hipótesis          : Corredor redundante: la avenida corre dentro de una retícula densa y conserva sus carriles auxiliares, así que el tráfico se reacomoda a una cuadra de distancia. Se espera un Δ% pequeño.


In [10]:
# La redundancia de la Reforma está en sus carriles auxiliares, que llevan otro
# nombre y por eso el escenario NO los cierra. Vale la pena dejarlo medido.
aux = red.ways_con_nodos(["Carril Auxiliar Avenida Reforma"])
print(f"Carriles auxiliares de la Reforma que quedan abiertos: "
      f"{len(aux)} ways, {sum(red.largo_km(w['geom']) for w in aux):.2f} km")
print("Esa es, literalmente, la redundancia que el escenario 'reforma' quiere medir.")

Carriles auxiliares de la Reforma que quedan abiertos: 17 ways, 4.20 km
Esa es, literalmente, la redundancia que el escenario 'reforma' quiere medir.


## 8. Mapa del área

In [11]:
import folium

centro = ((config.BBOX[0] + config.BBOX[2]) / 2, (config.BBOX[1] + config.BBOX[3]) / 2)
m = folium.Map(location=centro, zoom_start=14, tiles="OpenStreetMap")

# Caja de estudio.
folium.Rectangle(
    bounds=[(config.BBOX[0], config.BBOX[1]), (config.BBOX[2], config.BBOX[3])],
    color=config.GREY, weight=2, dash_array="6", fill=False,
).add_to(m)

# Red estructurante, de fondo.
for w in ways:
    if w["highway"] in ("motorway", "trunk", "primary") and len(w["geom"]) > 1:
        folium.PolyLine(w["geom"], color=config.GREY, weight=1.5, opacity=.45).add_to(m)

# Las dos candidatas.
colores = {"vista_hermosa": config.ORANGE, "reforma": config.NAVY}
for clave, ws in candidatas.items():
    for w in ws:
        if len(w["geom"]) > 1:
            folium.PolyLine(
                w["geom"], color=colores[clave], weight=5, opacity=.9,
                tooltip=f"{config.ESCENARIOS[clave]['etiqueta']} · way {w['way_id']}",
            ).add_to(m)

# Centros de zona y puntos ancla.
for k, v in config.ZONAS.items():
    folium.CircleMarker(v["centro"], radius=8, color=config.YELLOW,
                        fill=True, fill_opacity=.85, tooltip=v["nombre"]).add_to(m)
for k, p in config.PUNTOS.items():
    folium.CircleMarker(p, radius=4, color="black", fill=True,
                        fill_opacity=1, tooltip=k).add_to(m)

m.save(str(config.FIGURAS / "01_area_de_estudio.html"))
print(f"guardado: {config.FIGURAS / '01_area_de_estudio.html'}")
m

guardado: /Users/fabianprado/Documents/projects/2026 - 2 /modelacion/proyecto/figuras/01_area_de_estudio.html


## 9. Derivados

Lo que este notebook le pasa a los siguientes. Son archivos chicos y **sí** van
al repositorio, para que nadie tenga que volver a golpear Overpass solo para
saber cuántos ways tiene el Boulevard Vista Hermosa.

In [12]:
derivado = {
    "bbox": config.BBOX,
    "red_del_area": {
        "ways": len(ways),
        "km_totales": round(sum(kms.values()), 2),
        "km_por_clase": {h: round(kms[h], 2) for h in kms},
    },
    "candidatas": {
        clave: {
            "etiqueta": config.ESCENARIOS[clave]["etiqueta"],
            "nombres_osm": config.ESCENARIOS[clave]["nombres_osm"],
            "ways": len(ws),
            "way_ids": sorted(w["way_id"] for w in ws),
            "nodos": sum(len(w["nodos"]) for w in ws),
            "segmentos": len(red.pares_de_nodos(ws)),
            "km": round(sum(red.largo_km(w["geom"]) for w in ws), 2),
        }
        for clave, ws in candidatas.items()
    },
    "rutas_referencia": base_rutas.round(4).to_dict("records"),
    "servidor_usado": SERVIDOR,
}

p = red.guardar_derivado(derivado, "01_red.json")
print(f"guardado: {p}")

guardado: /Users/fabianprado/Documents/projects/2026 - 2 /modelacion/proyecto/data/derivados/01_red.json


---

## Qué queda listo

- Extracto de OSM descargado y grafo de OSRM construido (o instrucciones claras
  para construirlo).
- Servidor respondiendo, validado contra el público.
- La red del área caracterizada: **5 512 ways, ~745 km conducibles**.
- Las dos vías candidatas identificadas con sus IDs de way y sus nodos.
- `data/derivados/01_red.json` para los notebooks siguientes.

**Sigue:** [`02_linea_base_y_cierres.ipynb`](02_linea_base_y_cierres.ipynb) —
matriz origen-destino de referencia, generación de los archivos de cierre y las
tres verificaciones de que el cierre efectivamente cerró
([`estructura-proyecto.md` §12](../estructura-proyecto.md)).